# Applications Lab

```{contents}
:local:
:depth: 2
```


```{index} stack; queue; deque; choosing a data structure
```

## Choosing the Access Rule



Stacks, queues, and deques are easiest to choose when you name the rule for the next item:

| Situation | Next item should be | Structure |
|---|---|---|
| Undo the most recent edit | Most recently saved | Stack |
| Serve customers fairly | Earliest waiting | Queue |
| Process both urgent and normal work | Either front or back | Deque |

The access rule is more important than the English noun. A "list of tasks" might be a stack, a queue, or a deque depending on how tasks should be removed.


In [ ]:
using System;
using System.Collections.Generic;

var urgent = new Stack<string>();
urgent.Push("Restart failed service");
urgent.Push("Rollback broken update");

var normal = new Queue<string>();
normal.Enqueue("Update documentation");
normal.Enqueue("Clean old logs");

if (urgent.Count > 0)
{
    Console.WriteLine($"Handle now: {urgent.Pop()}");
}
else if (normal.Count > 0)
{
    Console.WriteLine($"Handle next: {normal.Dequeue()}");
}


This example uses two structures because there are two policies: urgent tasks use most-recent-first, and normal tasks use first-come-first-served.


```{index} backtracking; stack
```

## Backtracking with a Stack



Backtracking means trying a path, then returning to an earlier choice if that path does not work. A stack records the path because the most recent choice is the first one to undo.


In [ ]:
using System;
using System.Collections.Generic;

var path = new Stack<string>();
path.Push("Start");
path.Push("Hallway");
path.Push("Storage room");

Console.WriteLine($"Dead end at {path.Pop()}");
Console.WriteLine($"Return to {path.Peek()}");
path.Push("Server room");

Console.WriteLine("Current path:");
foreach (string location in path)
{
    Console.WriteLine(location);
}


`Stack<T>` enumeration visits the top first, so the displayed path starts with the current location and moves backward through recent choices.


```{index} breadth-first order; queue
```

## Breadth-First Processing with a Queue



Queues are central to breadth-first processing. When each item can discover more items, newly discovered items go to the back so earlier layers finish first.


In [ ]:
using System;
using System.Collections.Generic;

var pending = new Queue<string>();
pending.Enqueue("home");
var visited = new HashSet<string> { "home" };

var links = new Dictionary<string, string[]>
{
    ["home"] = new[] { "products", "support" },
    ["products"] = new[] { "pricing" },
    ["support"] = new[] { "faq", "contact" },
    ["pricing"] = Array.Empty<string>(),
    ["faq"] = Array.Empty<string>(),
    ["contact"] = Array.Empty<string>()
};

while (pending.Count > 0)
{
    string page = pending.Dequeue();
    Console.WriteLine(page);
    foreach (string next in links[page])
    {
        if (visited.Add(next)) pending.Enqueue(next);
    }
}


This is the same queue pattern used later for breadth-first graph traversal. The queue preserves discovery order across layers.


```{index} stack; operation cost; queue; operation cost
```

## Comparing Costs



For the restricted operations, these structures are usually designed to make the main access rule fast:

| Structure | Operation | Typical cost |
|---|---|---:|
| Growable array stack | Push / pop | O(1) amortized push; O(n) growth; O(1) pop |
| Bounded ring queue | Enqueue / dequeue | O(1); full enqueue rejects |
| Growable array queue | Enqueue / dequeue | O(1) amortized enqueue; O(n) growth; O(1) dequeue |
| Deque | Add/remove either end | O(1) with the right representation |
| List used as queue | Remove index 0 | O(n) |

The tempting implementation is not always the right one. Removing from the front of a plain array or list usually shifts many values.


In [ ]:
using System;
using System.Collections.Generic;

var slowQueue = new List<string> { "A", "B", "C" };
string first = slowQueue[0];
slowQueue.RemoveAt(0);

Console.WriteLine(first);
Console.WriteLine(string.Join(", ", slowQueue));


`RemoveAt(0)` is simple to read, but it moves the remaining elements left. `Queue<T>` communicates the intended operation and avoids relying on a slow pattern.


```{index} undo and redo require two histories
```

## Undo and Redo Require Two Histories



A realistic undo operation restores a prior state, rather than just printing an action name. Save the previous report state on undo before an edit. Undo saves the current state on redo and restores the latest undo state. Redo reverses that transfer.

| Request | Current state | Undo, bottom to top | Redo, bottom to top |
| --- | --- | --- | --- |
| Start | Draft | Empty | Empty |
| Edit Reviewed | Reviewed | Draft | Empty |
| Edit Approved | Approved | Draft, Reviewed | Empty |
| Undo | Reviewed | Draft | Approved |
| Redo | Approved | Draft, Reviewed | Empty |
| Undo; edit Revised | Revised | Draft, Reviewed | Empty |

A new edit after undo clears redo because it creates a new branch of history. This small editor stores complete immutable string snapshots. Large editors may store inverse commands or smaller changes to limit memory use.

**Predict:** Undo twice after the final edit. What states return? What should an extra undo do when its history is empty?


In [ ]:
using System;
using System.Collections.Generic;
string current = "Draft";
var undo = new Stack<string>();
var redo = new Stack<string>();
void Edit(string next)
{
    undo.Push(current); current = next; redo.Clear();
}
bool Undo()
{
    if (!undo.TryPop(out string? previous)) return false;
    redo.Push(current); current = previous; return true;
}
bool Redo()
{
    if (!redo.TryPop(out string? next)) return false;
    undo.Push(current); current = next; return true;
}
Edit("Reviewed"); Edit("Approved");
Undo(); Console.WriteLine($"after undo={current}");
Redo(); Console.WriteLine($"after redo={current}");
Undo(); Edit("Revised");
Console.WriteLine($"after edit={current}, redo available={Redo()}");


```{index} worklist order, duplicate discovery, and fairness
```

## Worklist Order, Duplicate Discovery, and Fairness



For the page-link example, after processing home the queue is `[products, support]`. Products adds pricing behind support; support then adds faq and contact. The processing order is home, products, support, pricing, faq, contact. Newly discovered pages join the back, so earlier layers finish first.

Mark a page discovered when it is enqueued. If both products and support link to pricing, only the first discovery enqueues it. Marking only when dequeuing can put duplicate pending entries into the queue. A visited set also prevents a link back to home from causing endless processing. The graph chapters develop the full traversal argument; here the important fact is the worklist policy.

Always serving urgent tickets before normal tickets may starve normal work if urgent arrivals never stop. A deque that adds urgency at the front changes service order; it does not guarantee fairness. A policy such as at most two urgent tickets before one waiting normal ticket makes a different promise.

**Design:** Trace that two-to-one policy for urgent U1, U2, U3 and normal N1, N2. Specify ordering within each group and behavior when either group is empty. What additional policy would you need if tickets have different processing times?


```{index} validate behavior before measuring
```

## Validate Behavior Before Measuring



Test each contract independently: stack LIFO, queue FIFO, deque both ends, and peek without removal. Include empty access, singleton removal and reuse, duplicates, default values, and failure without mutation. For a bounded queue, also test full rejection and wraparound after several cycles.

The same operations can be compared with `Stack<T>` or `Queue<T>` as reference models. Count should agree after each operation; removed values and peek results must agree too. Tests are evidence, while invariant reasoning explains why every legal operation preserves the access rule.

**Transfer:** A service desk needs a queue for waiting tickets and a stack for recently completed tickets that may be restored. Explain why these are two contracts. Describe how to restore a completed ticket and whether it returns to the front or back of the waiting line. State the fairness consequence before choosing a deque.


```{index} stack; queue; exercise
```

## Exercise



In [ ]:
// Exercise: Split urgent and normal tickets
// Urgent tickets should be handled most-recent-first.
// Normal tickets should be handled first-come-first-served.
// Print one urgent ticket if any exist; otherwise print one normal ticket.

using System;
using System.Collections.Generic;

var urgent = new Stack<string>();
var normal = new Queue<string>();

urgent.Push("U-1");
urgent.Push("U-2");
normal.Enqueue("N-1");
normal.Enqueue("N-2");

// Your code starts here.



// Your code ends here.


In [1]:
// Solution
using System;
using System.Collections.Generic;

var urgent = new Stack<string>();
var normal = new Queue<string>();

urgent.Push("U-1");
urgent.Push("U-2");
normal.Enqueue("N-1");
normal.Enqueue("N-2");

if (urgent.Count > 0)
{
    Console.WriteLine(urgent.Pop());
}
else if (normal.Count > 0)
{
    Console.WriteLine(normal.Dequeue());
}
else
{
    Console.WriteLine("No tickets waiting.");
}


U-2


```{index} summary
```

## Summary



Stacks, queues, and deques are small ADTs with big design value. Once the access rule is clear, the implementation and the C# library type usually become easier to choose.


```{rubric} Footnotes
```

[^1]: Breadth-first graph traversal is covered in more depth in later graph chapters; here the goal is to see why a queue fits the processing rule.
